In [1]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import time

# ─────────────────────────────────────────────
# 1. Load your trained Word2Vec model
# ─────────────────────────────────────────────
model_path = "./custom_w2v_groundtruth.model"
print("[INFO] Loading Word2Vec model...")
model = Word2Vec.load(model_path)
print("[INFO] Model loaded.")

# ─────────────────────────────────────────────
# 2. Load the inference result file
# ─────────────────────────────────────────────
# --- Load CSV file ---
base_path = "../../../"
goto_folder = "ResultGroup/1.Wigner/"
filename = "WignerRefactor-QuantumVLM-2.5VL-7B-v2444.csv"
print("[INFO] Loading inference CSV...")
df = pd.read_csv(f"{base_path}{goto_folder}{filename}")
print("[INFO] Rows:", len(df))

# define your column names:
PRED_COL = "generated"
GT_COL   = "ground_truth"

# ─────────────────────────────────────────────
# 3. Convert sentence to vector
# ─────────────────────────────────────────────
def sentence_vector(text):
    words = str(text).lower().split()
    vecs = [model.wv[w] for w in words if w in model.wv]

    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# ─────────────────────────────────────────────
# 4. Compute similarity with progress log
# ─────────────────────────────────────────────
scores = []
total = len(df)
last_print = time.time()

print("[INFO] Scoring...")

for i, row in df.iterrows():
    v1 = sentence_vector(row[PRED_COL])
    v2 = sentence_vector(row[GT_COL])

    sim = cosine_similarity([v1], [v2])[0][0]
    scores.append(sim)

    # print progress every second
    if time.time() - last_print > 1:
        pct = (i+1) / total * 100
        print(f"[INFO] {i+1}/{total} ({pct:.2f}%)")
        last_print = time.time()

df["w2v_score"] = scores

print("[INFO] Scoring complete.")

# ─────────────────────────────────────────────
# 5. Save to new CSV
# ─────────────────────────────────────────────
out_path = "output/1-2-3-2.word2vec-wigner-quantumvlm-2.5vl-v2.csv"
df.to_csv(out_path, index=False)

print("[INFO] Saved to:", out_path)

# ─────────────────────────────────────────────
# 6. Print mean score
# ─────────────────────────────────────────────
mean_score = df["w2v_score"].mean()
print("[INFO] Mean W2V score:", mean_score)


[INFO] Loading Word2Vec model...
[INFO] Model loaded.
[INFO] Loading inference CSV...
[INFO] Rows: 1086
[INFO] Scoring...
[INFO] Scoring complete.
[INFO] Saved to: output/1-2-3-2.word2vec-wigner-quantumvlm-2.5vl-v2.csv
[INFO] Mean W2V score: 0.94537127
